In [53]:
import os
print(os.getcwd())
print(os.listdir())

/Users/sanjanasubramanian/Documents/mexico-nearshoring/notebooks
['mexico_charts.ipynb']


In [54]:
import altair as alt
import pandas as pd
import numpy as np

pallete = {
    "shadow": "rgba(24, 42, 56, 0.1)",
    "nominal_1": "#179fdb",
    "nominal_2": "#e6224b",
    "nominal_3": "#f4c245",
    "nominal_4": "#122b39",
    "nominal_5": "#eb5c2e",
    "nominal_6": "#36b7b4",
    "Other_3": "#d6d4d4",
    "Deemphasize_Discrete": "rgba(24, 42, 56)",
    "Deemphasize_Other": "rgba(93, 115, 102, 1)",
    "Deemphasize_Continuous": "rgba(24, 42, 56, 0.2)",
    "accent": "#179fdb",
    "domain": "#122b39",
    "bar": {
        "accent_1": "#122b39",
        "other": "#a8c0de",
        "accent_2": "#eb5c2e",
    },
}

In [55]:
@alt.theme.register("report", enable=True)
def report():
    return alt.theme.ThemeConfig({
        "config": {
            "font": "Circular Std",
            "mark": {"line": {"interpolate": "linear"}},
            "view": {"stroke": "transparent", "width": 400, "height": 300},
            "range": {
                "category": [
                    pallete["nominal_1"], pallete["nominal_2"],
                    pallete["nominal_3"], pallete["nominal_4"],
                    pallete["nominal_5"], pallete["nominal_6"],
                ],
                "diverging": ["#E6224B","#E54753","#C9C9C9","#179FDB","#122B39"],
                "heatmap": ["#C9C9C9","#179FDB","#0063AF","#122B39"],
                "ordinal": ["#00A767","#36B7B4","#179FDB","#0063AF","#243B5A"],
            },
            "axisX": {
                "labelColor": pallete["domain"],
                "tickColor": pallete["domain"],
                "domainColor": pallete["domain"],
                "domainOpacity": 0.5,
                "gridOpacity": 0,
                "labelAngle": 0,
                "labelAlign": "center",
                "labelFontSize": 11,
                "labelPadding": 5,
                "tickCount": 10,
                "tickSize": 0,
                "title": "",
            },
            "axisY": {
                "labelColor": pallete["domain"],
                "tickColor": pallete["domain"],
                "domainColor": pallete["domain"],
                "gridColor": pallete["domain"],
                "gridDash": [1, 5],
                "gridOpacity": 0.5,
                "labelPadding": 5,
                "labelFontSize": 11,
                "domainOpacity": 0.5,
                "tickSize": 0,
                "title": None,
                "titleAlign": "left",
                "titleAngle": 0,
                "titleBaseline": "bottom",
                "titleColor": pallete["domain"],
                "titleOpacity": 0.9,
                "titleX": 0,
                "titleY": -7,
            },
        }
    })

In [56]:
# ── Read Chart 1 data ──
df1 = pd.read_excel("../data/Chartbook_Mexico_Nearshoring_ECO_v080626.xlsx",
                     sheet_name="Chart 1", skiprows=1,
                     usecols="A:C", names=["year","Exports to the US","Imports from the US"])

df1 = df1[df1["year"] <= 2025]
df1["Exports to the US"] = df1["Exports to the US"] / 1000
df1["Imports from the US"] = df1["Imports from the US"] / 1000

df1_long = df1.melt(id_vars=["year"], var_name="Series", value_name="Value")
df1_long["Date"] = pd.to_datetime(df1_long["year"], format="%Y")

# ── Chart 1: US-Mexico bilateral trade ──
# Filter from 1990
df1_plot = df1_long[df1_long["year"] >= 1990].copy()

# ── Chart 1: Stacked area ──
chart1 = alt.Chart(df1_plot).mark_area().encode(
    x=alt.X('Date:T', axis=alt.Axis()),
    y=alt.Y('Value:Q',
            axis=alt.Axis(title='US$ billions', format='.0f'),
            stack=True),
    color=alt.Color('Series:N', legend=alt.Legend(orient='top', title=None),
                    scale=alt.Scale(
                        domain=["Exports to the US","Imports from the US"],
                        range=[pallete["nominal_1"], pallete["nominal_2"]])),
    order=alt.Order('Series:N', sort='descending')
)

chart1




alt.Chart(...)

In [57]:
# ── Read Chart 2 data ──
df2 = pd.read_excel("../data/Chartbook_Mexico_Nearshoring_ECO_v080626.xlsx",
                     sheet_name="Chart 2", skiprows=1,
                     usecols="A:B", names=["Country","Value"])

df2 = df2.dropna()
df2 = df2[df2["Country"] != "Total general"]
df2["Value"] = df2["Value"] / 1000  # to billions

# ── Chart 2: Cumulative FDI by country of origin ──
chart2 = alt.Chart(df2).mark_bar().encode(
    x=alt.X('Value:Q', axis=alt.Axis(title='US$ billions', format='.0f')),
    y=alt.Y('Country:N', sort='-x'),
    color=alt.condition(
        alt.datum.Country == 'US',
        alt.value(pallete["nominal_1"]),
        alt.value(pallete["bar"]["other"])
    )
)

text = alt.Chart(df2).mark_text(
    align='left',
    dx=3,
    fontSize=11,
).encode(
    x=alt.X('Value:Q'),
    y=alt.Y('Country:N', sort='-x'),
    text=alt.Text('Value:Q', format='.1f')
)

chart2 = chart2 + text
chart2

alt.LayerChart(...)

In [58]:
# ── Chart 3: Stacked bar, imports by origin ──
country_order = ["US","China","Taiwan","South Korea","Vietnam","Rest of countries"]

chart3 = alt.Chart(df3_long).mark_bar(width=80).encode(
    x=alt.X('Year:N', axis=alt.Axis(title=None, labelAngle=0)),
    y=alt.Y('Value:Q', axis=alt.Axis(title='US$ billions', format='.0f'),
            stack=True),
    color=alt.Color('Country:N',
                    legend=alt.Legend(orient='top', title=None, columns=3),
                    scale=alt.Scale(
                        domain=country_order,
                        range=[pallete["nominal_1"], pallete["nominal_2"],
                               pallete["nominal_3"], pallete["nominal_4"],
                               pallete["nominal_5"], pallete["nominal_6"]])),
    order=alt.Order('country_sort:Q'),
).transform_calculate(
    country_sort="indexof(['US','China','Taiwan','South Korea','Vietnam','Rest of countries'], datum.Country)"
)

chart3

NameError: name 'df3_long' is not defined

In [ ]:
# ── Read Chart 4 data ──
df4 = pd.read_excel("../data/Chartbook_Mexico_Nearshoring_ECO_v080626.xlsx",
                     sheet_name="Chart 4", skiprows=1,
                     usecols="B:F,H:I", header=None,
                     names=["year","FDI","New investment","Reinvestment",
                            "Parent company","New inv share","Reinv share"],
                     nrows=9)

# Convert shares to 0-1 scale (they're in %)
df4["New inv share"] = df4["New inv share"] / 100
df4["Reinv share"] = df4["Reinv share"] / 100

# Convert levels to billions
for col in ["FDI","New investment","Reinvestment","Parent company"]:
    df4[col] = df4[col] / 1000

df4["Date"] = pd.to_datetime(df4["year"], format="%Y")

# Long format for levels (lines)
df4_levels = df4.melt(
    id_vars=["Date","year"],
    value_vars=["New investment","Reinvestment","Parent company"],
    var_name="Series", value_name="Value"
)

# Long format for shares (markers)
df4_shares = df4.melt(
    id_vars=["Date","year"],
    value_vars=["New inv share","Reinv share"],
    var_name="Series", value_name="Value"
)
df4_shares["Series"] = df4_shares["Series"].replace({
    "New inv share": "New investment",
    "Reinv share": "Reinvestment",
})

In [ ]:
# ── Chart 4: FDI by type, dual axis ──

# Filter levels to just two series (no parent company)
df4_lines = df4_levels[df4_levels["Series"] != "Parent company"]

color_scale = alt.Scale(
    domain=["New investment","Reinvestment"],
    range=[pallete["nominal_1"], pallete["nominal_2"]]
)

# Smooth lines: levels (left axis)
lines = alt.Chart(df4_lines).mark_line(interpolate='monotone').encode(
    x=alt.X('Date:T', axis=alt.Axis()),
    y=alt.Y('Value:Q', axis=alt.Axis(title='US$ billions (lines)', format='.0f')),
    color=alt.Color('Series:N', legend=alt.Legend(orient='top', title=None),
                    scale=color_scale)
)

# Markers only: shares (right axis)
share_dots = alt.Chart(df4_shares).mark_point(size=60, filled=True).encode(
    x=alt.X('Date:T'),
    y=alt.Y('Value:Q',
            axis=alt.Axis(title='% of total FDI (markers)', format='.0%'),
            scale=alt.Scale(domain=[0, 0.85])),
    color=alt.Color('Series:N', legend=None, scale=color_scale),
    shape=alt.Shape('Series:N', legend=alt.Legend(orient='top', title=None),
                    scale=alt.Scale(
                        domain=["New investment","Reinvestment"],
                        range=["square","diamond"]))
)

chart4 = alt.layer(lines, share_dots).resolve_scale(y='independent')
chart4

alt.LayerChart(...)

In [ ]:
chart1.save("../charts/chart1_bilateral_trade.json")
chart2.save("../charts/chart2_fdi_by_origin.json")
chart3.save("../charts/chart3_imports_by_origin.json")
chart4.save("../charts/chart4_fdi_by_type.json")

In [ ]:
import os
os.makedirs("../charts", exist_ok=True)

for name, chart in [("chart1_bilateral_trade", chart1),
                     ("chart2_fdi_by_origin", chart2),
                     ("chart3_imports_by_origin", chart3),
                     ("chart4_fdi_by_type", chart4)]:
    chart.save(f"../charts/{name}.json")
    chart.save(f"../charts/{name}.png", scale_factor=2.0)

print("Saved all charts")

Saved all charts
